In [1]:
import requests
import time
import random
import string
import tiktoken
from concurrent.futures import ThreadPoolExecutor, as_completed

# ==========================================
# [설정 영역]
# ==========================================
MODEL_NAME = "qwen3:8b"
API_URL    = "http://localhost:11434/api/generate"

# 동시 요청 수 목록 — 각 값에 대해 독립적으로 벤치마크 수행
CONCURRENT_REQUESTS = [1, 2, 4, 16, 64]

# 각 (프롬프트 크기, 동시 요청 수) 조합당 반복 횟수
ITERATIONS = 10

TEST_CASES = [
    {"label": "128",  "target_tokens": 128,   "output_len": 300},
    {"label": "1K",   "target_tokens": 1024,  "output_len": 300},
    {"label": "4K",   "target_tokens": 4096,  "output_len": 300},
]

MAX_CONTEXT = 16384
tokenizer   = tiktoken.get_encoding("cl100k_base")

In [2]:
# ==========================================
# [프롬프트 생성]
# ==========================================
def generate_exact_prompt(target_tokens):
    """
    tiktoken을 사용하여 정확히 target_tokens 길이에 맞게 프롬프트를 생성합니다.
    캐시 무력화를 위해 매 호출마다 다른 nonce가 삽입됩니다.
    """
    if target_tokens < 10:
        return "hi " * (target_tokens // 2 + 1)

    nonce  = "".join(random.choices(string.ascii_letters + string.digits, k=16))
    prefix = f"System Nonce: {nonce}\n\n"
    prefix_tokens = tokenizer.encode(prefix)

    if target_tokens <= len(prefix_tokens):
        return prefix

    remaining_target = target_tokens - len(prefix_tokens)
    base_text   = "The deepseek coder is a powerful model for coding tasks and general reasoning benchmarks. It efficiently handles context and generation tasks with mixture of experts architecture. "
    base_tokens = tokenizer.encode(base_text)

    repeat_count   = (remaining_target // len(base_tokens)) + 1
    extended_tokens = (base_tokens * repeat_count)[:remaining_target]

    return tokenizer.decode(prefix_tokens + extended_tokens)

In [3]:
# ==========================================
# [단일 요청]
# ==========================================
def run_single_test(case):
    """
    스트리밍 모드로 요청을 실행하고 wall-clock 기준의 TTFT로 prefill/decode 속도를 측정합니다.
    vllm-concurrent-test.ipynb와 동일한 측정 방식으로, 대기열 지연이 수치에 반영됩니다.
    실패 시 (None, None, None, error_message) 를 반환합니다.
    """
    import json
    prompt = generate_exact_prompt(case["target_tokens"])

    payload = {
        "model":  MODEL_NAME,
        "prompt": prompt,
        "stream": True,
        "options": {
            "num_predict": case["output_len"],
            "num_ctx":     MAX_CONTEXT,
            "temperature": 0.0,
            "num_gpu":     99,
        }
    }

    try:
        start_time       = time.time()
        first_token_time = None
        p_count          = 0
        e_count          = 0

        with requests.post(API_URL, json=payload, stream=True, timeout=600) as resp:
            for line in resp.iter_lines():
                if not line:
                    continue
                data = json.loads(line)

                if data.get("error"):
                    return None, None, None, f"Error: {data['error']}"

                # 첫 번째 토큰 도착 시각 기록
                if data.get("response") and first_token_time is None:
                    first_token_time = time.time()

                # 마지막 청크: 서버 집계 토큰 수 수집
                if data.get("done"):
                    p_count = data.get("prompt_eval_count", 0)
                    e_count = data.get("eval_count", 0)

        end_time = time.time()

        prefill_dur = (first_token_time - start_time) if first_token_time else 0
        decode_dur  = (end_time - first_token_time)   if first_token_time else 0

        prefill = p_count / prefill_dur if prefill_dur > 0 else 0
        decode  = (e_count - 1) / decode_dur if (decode_dur > 0 and e_count > 1) else 0

        return prefill, decode, p_count, e_count

    except requests.exceptions.ConnectionError:
        return None, None, None, "CUDA OOM or Server Down"
    except Exception as e:
        return None, None, None, str(e)

In [4]:
# ==========================================
# [동시 요청 배치]
# ==========================================
def run_concurrent_batch(case, n_concurrent):
    """
    n_concurrent 개의 요청을 동시에 실행합니다.

    반환값:
        results      : 성공한 요청들의 (prefill, decode, p_count, e_count) 리스트
        wall_elapsed : 배치 전체 경과 시간(초) — wall-clock 기준
        n_failed     : 실패한 요청 수
    """
    results  = []
    n_failed = 0

    wall_start = time.time()
    with ThreadPoolExecutor(max_workers=n_concurrent) as executor:
        futures = [executor.submit(run_single_test, case) for _ in range(n_concurrent)]
        for future in as_completed(futures):
            prefill, decode, p_count, e_count = future.result()
            if prefill is None:
                n_failed += 1
            else:
                results.append((prefill, decode, p_count, e_count))
    wall_elapsed = time.time() - wall_start

    return results, wall_elapsed, n_failed

In [5]:
# ==========================================
# [메인 벤치마크]
# ==========================================
print(f"\n🚀 Concurrent Benchmark: {MODEL_NAME}")
print(f"📦 Max Context: {MAX_CONTEXT} | 🔄 Iterations: {ITERATIONS} | ⚡ Concurrency levels: {CONCURRENT_REQUESTS}")
print("-" * 121)
print(f"{'Prompt Size':<15} | {'Concurrency':>11} | {'Avg Prefill (t/s)':>17} | {'Avg Decode (t/s)':>16} | {'Avg Total (s)':>13} | {'Throughput (t/s)':>16} | {'Status'}")
print("-" * 121)

# Warm-up
print(f"{'Warm-up...':<15} | {'':>11} | {'...':>17} | {'...':>16} | {'...':>13} | {'...':>16} | Loading model")
requests.post(API_URL, json={"model": MODEL_NAME, "prompt": "hi", "stream": False}, timeout=120)

for case in TEST_CASES:
    for n_concurrent in CONCURRENT_REQUESTS:

        iter_prefills    = []
        iter_decodes     = []
        iter_totals      = []
        iter_throughputs = []
        any_error        = None

        for _ in range(ITERATIONS):
            results, wall_elapsed, n_failed = run_concurrent_batch(case, n_concurrent)

            if not results:  # 모든 요청 실패
                any_error = "All requests failed"
                break

            avg_prefill = sum(r[0] for r in results) / len(results)
            avg_decode  = sum(r[1] for r in results) / len(results)
            throughput  = sum(r[3] for r in results) / wall_elapsed

            iter_prefills.append(avg_prefill)
            iter_decodes.append(avg_decode)
            iter_totals.append(wall_elapsed)
            iter_throughputs.append(throughput)

            time.sleep(0.5)

        if any_error:
            print(f"{case['label']:<15} | {n_concurrent:>11} | {'Fail':>17} | {'Fail':>16} | {'Fail':>13} | {'Fail':>16} | {any_error}")
        else:
            final_prefill    = sum(iter_prefills)    / len(iter_prefills)
            final_decode     = sum(iter_decodes)     / len(iter_decodes)
            final_total      = sum(iter_totals)      / len(iter_totals)
            final_throughput = sum(iter_throughputs) / len(iter_throughputs)
            n_ok = n_concurrent - n_failed  # 마지막 배치 기준 성공 수
            status = "ok" if n_failed == 0 else f"ok"

            print(f"{case['label']:<15} | {n_concurrent:>11} | {final_prefill:>17.2f} | {final_decode:>16.2f} | {final_total:>13.2f} | {final_throughput:>16.2f} | {status}")

print("-" * 121)
print("✅ Benchmark Completed.")


🚀 Concurrent Benchmark: qwen3:8b
📦 Max Context: 16384 | 🔄 Iterations: 10 | ⚡ Concurrency levels: [1, 2, 4, 16, 64]
-------------------------------------------------------------------------------------------------------------------------
Prompt Size     | Concurrency | Avg Prefill (t/s) | Avg Decode (t/s) | Avg Total (s) | Throughput (t/s) | Status
-------------------------------------------------------------------------------------------------------------------------
Warm-up...      |             |               ... |              ... |           ... |              ... | Loading model
128             |           1 |              8.96 |           203.76 |          8.55 |            35.28 | ok
128             |           2 |              8.41 |           282.59 |         14.17 |            42.07 | ok
128             |           4 |              6.20 |           394.58 |         26.09 |            45.99 | ok
128             |          16 |              2.44 |           323.53 |         9